In [1]:
import math

In [2]:
class Online_Search_Problem():
    def GOAL_TEST(self, state):
        pass
    
    def ACTIONS(self, state):
        pass
    
    def COST(s,a,ss):
        pass

### 1. Online search

In [3]:
def ONLINE_DFS_AGENT(problem, c_state):
    
    global p_state, a, result, untried, unbacktracked
    
    #c_state là mục tiêu thì stop
    if problem.GOAL_TEST(c_state):
        result[(p_state, a)] = c_state
        return 'stop'
    
    #Trạng thái hiện tại là trạng thái mới
    if c_state not in untried:
        untried[c_state] = problem.ACTIONS(c_state)
    
    #Trạng thái trước đó khác none
    if p_state is not None:
        result[(p_state, a)] = c_state
        #Nếu key: c_state chưa có trong unbacktracked thì khỏi tạo nó với value = danh sách rỗng
        if c_state not in unbacktracked:
            unbacktracked[c_state] = []
        #Thêm p_state vào đầu danh sách các trạng thái chưa được quay lại của c_state
        unbacktracked[c_state].insert(0, p_state)
    
    #Không còn hành động nào để thử ở trạng thái hiện tại
    if not untried[c_state]:
        #Không còn trạng thái nào để quay lại
        if not unbacktracked[c_state]:
            return 'stop'
        else:
            b = None
            #Tìm hành động để c_state quay lại trạng thái chưa được quay lại
            for action in problem.ACTIONS(c_state):
                if result.get((c_state, action)) == unbacktracked[c_state][-1]:
                    b = action
                    break
            a = b
            unbacktracked[c_state].pop()
    else:
        a = untried[c_state].pop()
    
    p_state = c_state
    return a

### 2. LRTA*

In [4]:
def LRTAStar_COST(problem,s,a,neighbor):
        if neighbor is None:
            return h[s]
        else:
            return problem.COST(s,a,neighbor) + H[neighbor]

def LRTAStar_AGENT(problem, c_state):
    global p_state, a, result, H, h
    
    #Trạng thái hiện tại là mục tiêu
    if problem.GOAL_TEST(c_state):
        result[(p_state, a)] = c_state
        return 'stop'
    
    #c_state là trạng thái mới
    if c_state not in H:
        #Khởi tạo H(c_state) = heuristic(c_state)
        H[c_state] = h[c_state]
    
    #Nếu trạng thái trước đó khác none
    if p_state is not None:
        result[(p_state, a)] = c_state
        p_state_actions = problem.ACTIONS(p_state)
        min = math.inf
        
        #Tìm giá trị tốt nhất của H(p_state) (nhỏ nhất) thông qua các hàng xóm
        for action in p_state_actions:
            c = LRTAStar_COST(problem,p_state,action,result.get((p_state,action)))
            if c < min:
                min = c
        H[p_state] = min 
    
    c_state_actions = problem.ACTIONS(c_state)
    min = math.inf
    
    #Tìm ra hành động để đến được node hàng xóm, nơi trạng thái H(c_state) nhỏ nhất
    for action in c_state_actions:
        c = LRTAStar_COST(problem,c_state,action,result.get((c_state,action)))
        if c < min:
            min = c
            a = action
            
    p_state = c_state
    return a

### 3. Define the problem class

In [5]:
class GraphProblem(Online_Search_Problem):
    def __init__(self, graph, goal_state):
        self.graph = graph
        self.goal_state = goal_state
    
    def GOAL_TEST(self, state):
        return state == self.goal_state
    
    def ACTIONS(self, state):
        return [i for i, connected in enumerate(self.graph[state]) if connected]
    
    def COST(self,s,a,ss):
        return 1
        
    def TRANSITION(self,s,a):
        return a

### 4. Use the DFS online search and LRTA* algorithm defined above for the shortest path finding problem on the graph

#### 4.1 DFS Online search

In [6]:
graph = [[0, 1, 1, 0, 0],
         [1, 0, 0, 1, 1],
         [1, 0, 0, 1, 0],
         [0, 1, 1, 0, 1],
         [0, 1, 0, 1, 0]]
goal_state = 4
start_state = 0
problem = GraphProblem(graph, goal_state)

result = {}
untried = {}
unbacktracked = {}
p_state = None
a = None
c_state = start_state

while True:
    a = ONLINE_DFS_AGENT(problem, c_state)
    if a == 'stop':
        break
    next_state = problem.TRANSITION(c_state,a)
    c_state = next_state

print("Result:",result)
print("Path found:", end=" ")
path = [goal_state]
while path[-1] != start_state:
    for action, state in result.items():
        if state == path[-1]:
            path.append(action[0])
            break
print(*reversed(path))

Result: {(0, 2): 2, (2, 3): 3, (3, 4): 4}
Path found: 0 2 3 4


#### 4.2 LRTA*

In [7]:
graph = [[0, 1, 1, 0, 0],
         [1, 0, 0, 1, 1],
         [1, 0, 0, 1, 0],
         [0, 1, 1, 1, 1],
         [0, 1, 0, 1, 0]]
goal_state = 4
start_state = 0
problem = GraphProblem(graph, goal_state)

result = {}
H = {}
h = {0:5,
    1:1,
    2:3,
    3:2,
    4:0}
p_state = None
a = None
c_state = start_state

while True:
    a = LRTAStar_AGENT(problem, c_state)
    if a == 'stop':
        break
    next_state = problem.TRANSITION(c_state,a)
    c_state = next_state
    
print("Result:",result)
print("Path found:", end=" ")
path = [goal_state]
while path[-1] != start_state:
    for action, state in result.items():
        if state == path[-1]:
            path.append(action[0])
            break
print(*reversed(path))

Result: {(0, 1): 1, (1, 0): 0, (1, 3): 3, (3, 1): 1, (1, 4): 4}
Path found: 0 1 4
